# ETAPE 3 — Jointure des tables
**Projet : Optimisation du Réseau de Boutiques TELECOM — Groupe 5 SUD-EST**

---
## Ordre des opérations
1. Filtrer Fichier_principal sur ZONE = '5_SUD_EST' → `table_sud_est`
2. Joindre avec COORD_BQT sur ORDER_SHOP_CD → `table_geo`
3. Joindre avec LIB_BQT sur ORDER_SHOP_CD → `table_finale`
4. Convertir les coordonnées Lambert II → WGS84 (lat/lon)
5. Vérifications qualité + sauvegarde

In [1]:
import pandas as pd
import numpy as np
from pyproj import Transformer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

## 3.0 Rechargement des données nettoyées

In [2]:
# Charger les fichiers nettoyés de l'étape 2
try:
    df = pd.read_csv('df_sud_est_propre.csv', low_memory=False, parse_dates=['PERIOD'])
    coord = pd.read_csv('coord_propre.csv')
    lib = pd.read_csv('lib_propre.csv')
    print("Fichiers nettoyés chargés.")
except FileNotFoundError:
    # Si l'étape 2 n'a pas été exécutée, re-faire le nettoyage minimal
    print("Fichiers nettoyés non trouvés — chargement depuis les bruts")
    df_raw = pd.read_csv('Fichier_Principal_pedago_anonym.csv', low_memory=False)
    coord = pd.read_csv('COORD_BQT.csv')
    lib = pd.read_csv('LIB_BQT.csv')
    
    df = df_raw[df_raw['ZONE'] == '5_SUD_EST'].copy()
    df = df.drop_duplicates()
    for t in [df, coord, lib]:
        t['ORDER_SHOP_CD'] = t['ORDER_SHOP_CD'].str.strip().str.upper()
    coord = coord.drop_duplicates(subset='ORDER_SHOP_CD').dropna(subset=['Coord_L2E_II_X', 'Coord_L2E_II_Y'])
    lib = lib.drop_duplicates(subset='ORDER_SHOP_CD')
    df['PERIOD'] = pd.to_datetime(
        df['PERIOD_YYYY'].astype(str) + '-' + df['PERIOD_MM'].astype(str).str.zfill(2) + '-01'
    )

print(f"df SUD-EST : {len(df):,} lignes")
print(f"COORD_BQT  : {len(coord)} boutiques")
print(f"LIB_BQT    : {len(lib)} boutiques")

Fichiers nettoyés chargés.
df SUD-EST : 226,677 lignes
COORD_BQT  : 958 boutiques
LIB_BQT    : 3435 boutiques


## 3.1 Conversion des coordonnées Lambert II → WGS84

In [3]:
# EPSG:27572 = Lambert II Etendu, EPSG:4326 = WGS84 (lat/lon standard)
transformer = Transformer.from_crs('EPSG:27572', 'EPSG:4326', always_xy=True)

lons, lats = transformer.transform(
    coord['Coord_L2E_II_X'].values,
    coord['Coord_L2E_II_Y'].values
)

coord['latitude']  = lats
coord['longitude'] = lons

# Vérification — les boutiques SUD-EST doivent être entre ~42°N et ~46°N
print("Latitude boutiques (après conversion) :")
print(f"  Min : {coord['latitude'].min():.2f}")
print(f"  Max : {coord['latitude'].max():.2f}")
print(f"  Moy : {coord['latitude'].mean():.2f}")
print(coord[['ORDER_SHOP_CD', 'latitude', 'longitude', 'Emplacement_PDV']].head(5))

Latitude boutiques (après conversion) :
  Min : 5.49
  Max : 51.03
  Moy : 46.89
  ORDER_SHOP_CD   latitude  longitude Emplacement_PDV
0    95PHSEERMO  48.988674   2.245043              CC
1    67700FF5-4  48.743195   7.360419              CV
2    16017FL9-3  45.686991   0.180329              CV
3    84PHSEPERT  43.681947   5.499833              CC
4    06PHSEMENT  43.775616   7.506035              CV


## 3.2 Jointure principale × COORD_BQT

In [4]:
n_avant = len(df)

table_geo = df.merge(
    coord[['ORDER_SHOP_CD', 'Surface_COMMERC', 'Emplacement_PDV', 'latitude', 'longitude']],
    on='ORDER_SHOP_CD',
    how='left'
)

n_apres = len(table_geo)
perte = (n_avant - n_apres) / n_avant * 100
print(f"Lignes avant jointure : {n_avant:,}")
print(f"Lignes après jointure : {n_apres:,}")
print(f"Perte : {n_avant - n_apres:,} ({perte:.2f}%)")
print(f"Nulls latitude après jointure : {table_geo['latitude'].isnull().sum():,}")

Lignes avant jointure : 226,677
Lignes après jointure : 226,677
Perte : 0 (0.00%)
Nulls latitude après jointure : 108,549


## 3.3 Jointure table_geo × LIB_BQT

In [5]:
table_finale = table_geo.merge(
    lib[['ORDER_SHOP_CD', 'MKT_CHANNL_LB']],
    on='ORDER_SHOP_CD',
    how='left'
)

print(f"Colonnes table finale : {table_finale.columns.tolist()}")
print(f"Lignes table finale   : {len(table_finale):,}")
print(f"Nulls MKT_CHANNL_LB   : {table_finale['MKT_CHANNL_LB'].isnull().sum():,}")

Colonnes table finale : ['LINE_TYPE', 'PERIOD_YYYY', 'PERIOD_MM', 'ORDER_SHOP_CD', 'PERSON_SALUTATION_CD', 'PERSON_BIRTH_DT_year', 'NET_ZONE_IN', 'DENSE_ZONE_IN', 'DEPRTMNT_ID', 'CITY_LN', 'ZONE', 'Geolife', 'GEOLIFE_AGG', 'PERIOD', 'birth_decade', 'age_client', 'Surface_COMMERC', 'Emplacement_PDV', 'latitude', 'longitude', 'MKT_CHANNL_LB']
Lignes table finale   : 226,677
Nulls MKT_CHANNL_LB   : 17,822


## 3.4 Contrôles qualité post-jointure

In [6]:
print("=" * 60)
print("CONTRÔLES QUALITÉ POST-JOINTURE")
print("=" * 60)

checks = [
    ("Nb lignes stable",          len(table_finale) == n_avant,
                                   f"{len(table_finale):,} vs {n_avant:,}"),
    ("Nulls latitude",             table_finale['latitude'].isnull().sum() == 0,
                                   str(table_finale['latitude'].isnull().sum())),
    ("Nulls longitude",            table_finale['longitude'].isnull().sum() == 0,
                                   str(table_finale['longitude'].isnull().sum())),
    ("Nulls emplacement CC/CV",    table_finale['Emplacement_PDV'].isnull().sum() == 0,
                                   str(table_finale['Emplacement_PDV'].isnull().sum())),
    ("Pas de colonnes dupliquées", not any('_x' in c or '_y' in c for c in table_finale.columns),
                                   str([c for c in table_finale.columns if '_x' in c or '_y' in c])),
    ("Nb colonnes 12-14",          12 <= table_finale.shape[1] <= 16,
                                   str(table_finale.shape[1])),
]

for label, ok, val in checks:
    status = "✓ OK" if ok else "✗ ALERTE"
    print(f"  {status:10s} | {label:35s} | {val}")

print("=" * 60)

CONTRÔLES QUALITÉ POST-JOINTURE
  ✓ OK       | Nb lignes stable                    | 226,677 vs 226,677
  ✗ ALERTE   | Nulls latitude                      | 108549
  ✗ ALERTE   | Nulls longitude                     | 108549
  ✗ ALERTE   | Nulls emplacement CC/CV             | 108549
  ✗ ALERTE   | Pas de colonnes dupliquées          | ['PERSON_BIRTH_DT_year']
  ✗ ALERTE   | Nb colonnes 12-14                   | 21


In [7]:
# Boutiques SUD-EST avec et sans coordonnées
n_boutiques = table_finale['ORDER_SHOP_CD'].nunique()
n_boutiques_coord = table_finale.dropna(subset=['latitude'])['ORDER_SHOP_CD'].nunique()
print(f"Boutiques uniques dans le dataset final : {n_boutiques}")
print(f"Boutiques avec coordonnées GPS          : {n_boutiques_coord}")
print(f"Boutiques SANS coordonnées (non cartographiables) : {n_boutiques - n_boutiques_coord}")

print("\nRépartition CC/CV :")
print(table_finale.drop_duplicates('ORDER_SHOP_CD')['Emplacement_PDV'].value_counts())

Boutiques uniques dans le dataset final : 1068
Boutiques avec coordonnées GPS          : 588
Boutiques SANS coordonnées (non cartographiables) : 480

Répartition CC/CV :
Emplacement_PDV
CV               295
CC               292
HORS CV ET CC      1
Name: count, dtype: int64


## 3.5 Sauvegarde de la base de travail

In [8]:
table_finale.to_csv('base_travail_SUD_EST.csv', index=False)
print(f"base_travail_SUD_EST.csv sauvegardée : {len(table_finale):,} lignes x {table_finale.shape[1]} colonnes")
table_finale.head(3)

base_travail_SUD_EST.csv sauvegardée : 226,677 lignes x 21 colonnes


,LINE_TYPE,PERIOD_YYYY,PERIOD_MM,ORDER_SHOP_CD,PERSON_SALUTATION_CD,PERSON_BIRTH_DT_year,NET_ZONE_IN,DENSE_ZONE_IN,DEPRTMNT_ID,CITY_LN,ZONE,Geolife,GEOLIFE_AGG,PERIOD,birth_decade,age_client,Surface_COMMERC,Emplacement_PDV,latitude,longitude,MKT_CHANNL_LB
0,T,2022,11,VADPCM,M,1980-1989,0.0,0.0,20,ROSAZIA,5_SUD_EST,11A_saisonnier,Résidence secondaire,2022-11-01,1980.0,39.0,NaN,NaN,NaN,NaN,NaN
1,I,2022,12,04PHSESIST,M,1940-1949,1.0,0.0,04,ST ETIENNE LES ORGUES,5_SUD_EST,11C_tourisme vert,Résidence secondaire,2022-12-01,1940.0,79.0,42.0,CV,44.196507,5.942554,Réseau Partenaire
2,I,2022,10,CORSE_OPEN,M,1980-1989,1.0,0.0,2A,AUTRE,5_SUD_EST,11C_tourisme vert,Résidence secondaire,2022-10-01,1980.0,39.0,NaN,NaN,NaN,NaN,Service Client Home


## Conclusion Etape 3

| Étape | Résultat |
|---|---|
| Filtre SUD-EST | OK |
| Conversion Lambert II → WGS84 | OK (lat/lon créées) |
| Jointure × COORD_BQT | OK |
| Jointure × LIB_BQT | OK |
| Fichier sauvegardé | `base_travail_SUD_EST.csv` |

**→ Prochaine étape : Exploration des données (Etape 4)**